In [1]:

import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)



In [2]:
if os.path.isdir('FilmDamageSimulator'):
    print('FilmDamageSimulator already cloned, skipping.')
else:
    !git clone --depth 1 https://github.com/daniela997/FilmDamageSimulator.git

Cloning into 'FilmDamageSimulator'...
remote: Enumerating objects: 6141, done.
remote: Counting objects: 100% (6141/6141), done.
remote: Compressing objects: 100% (6131/6131), done.
remote: Total 6141 (delta 9), reused 6139 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (6141/6141), 197.02 MiB | 42.07 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Updating files: 100% (6137/6137), done.


In [3]:
%%writefile FilmDamageSimulator/damage_generator/generate_synthetic_only.py
"""
Generate damage overlay masks using ONLY the pre-classified synthetic damage
patches in /synthetic/<type>/ (e.g. scratches, smut), without ever touching
the real scanned film frames in /scans/.

This bypasses damage_generator.py's default behaviour, which always loads
/scans/ and mixes real scanned artifact crops into the sampling pool even
when --synthetic is passed. Here, only the folder(s) you name are loaded,
and artifact count/size statistics are fit on those patches' own area
distribution instead of the real-scan-derived Gamma distributions.

Usage:
    python generate_synthetic_only.py --types scratches,smut --height 1024 --width 1024
    python generate_synthetic_only.py --types scratches --procedural-scratches
    python generate_synthetic_only.py --types dirt --out-dir ./masks/dirt --size-variety-min 0.5 --size-variety-max 1.8
"""

import os
import argparse
import uuid
import random
import numpy as np
import pandas as pd
import cv2 as cv
import scipy.stats as stats
import skimage.transform as skimage_tf

from scans import load_images
from generate_masks import generate_perlin_noise_2d, increase_contrast, random_perlin_with_numpy, line_scratch


def sample_size_from_own_distribution(df, num_artifact):
    """Fit a Gamma distribution to this dataframe's OWN artifact areas
    (instead of a real-scan-derived one) and sample target sizes from it."""
    areas = df['Contour Area']
    gamma_param = stats.gamma.fit(areas, floc=0)
    shape, _, scale = gamma_param
    return np.random.gamma(shape, scale, num_artifact)


def sample_closest_in_area(df, target_areas):
    df = df.sample(frac=1).reset_index(drop=True)
    areas = df['Contour Area']
    indexes = []
    for target in target_areas:
        candidates = df.iloc[(areas - target).abs().argsort()[:15]].index.tolist()
        index = random.choice(candidates)
        indexes.append(index)
        areas = areas.drop(areas.index[[index]])
    picked = df.iloc[indexes].copy()
    picked['Target size'] = target_areas
    return picked


def build_mask(target_size, per_type_dfs, per_type_counts, rescale=True, verbose=False,
                size_variety=(0.5, 1.8), intensity_boost=1.0):
    rescale_factor = (target_size[0] / 2560 if target_size[0] <= target_size[1]
                       else target_size[1] / 2560) if rescale else 1.

    selected_frames = []
    for artifact_type, df in per_type_dfs.items():
        lo, hi = per_type_counts[artifact_type]
        num = int(np.random.randint(lo, hi + 1))
        if num == 0 or len(df) == 0:
            continue
        target_areas = sample_size_from_own_distribution(df, num)
        picked = sample_closest_in_area(df, target_areas)
        selected_frames.append(picked)
        if verbose:
            print(f"Selected {num} '{artifact_type}' artifacts")

    if not selected_frames:
        raise ValueError("No artifacts selected - check your --types and --min-count/--max-count")

    selected_artifacts_df = pd.concat(selected_frames, ignore_index=True)
    artifacts_num = len(selected_artifacts_df)

    mask_final = np.zeros(target_size).astype(np.uint8)
    perlin_noise = generate_perlin_noise_2d(target_size, (2, 2))
    normalised_noise = (perlin_noise - np.min(perlin_noise)) / np.ptp(perlin_noise)
    xs, ys = random_perlin_with_numpy(artifacts_num, normalised_noise)
    random_angles = np.random.randint(0, 360, size=artifacts_num)

    i = 0
    for _, artifact_row in selected_artifacts_df.iterrows():
        try:
            artifact = artifact_row['Artifact'].astype(np.uint8)
            random_scale = artifact_row['Target size'] / artifact_row['Contour Area']
            random_angle = random_angles[i]
            # size_variety adds an independent random multiplier on top of the
            # gamma-fit-based scale above -- without this, size variety is
            # entirely bounded by whatever area distribution the raw artifact
            # patches happen to have, which can look narrower than intended if
            # the patch library itself is fairly uniform in size.
            variety_factor = np.random.uniform(size_variety[0], size_variety[1])
            new_rescale_factor = rescale_factor * np.sqrt(random_scale) * variety_factor
            artifact = skimage_tf.rescale(artifact, round(new_rescale_factor, 2), anti_aliasing=True, preserve_range=True)
            artifact = skimage_tf.rotate(artifact, angle=random_angle, resize=True, preserve_range=True)
            artifact_w, artifact_h = artifact.shape[:2]

            x1 = xs[i] - artifact_w // 2
            x2 = x1 + artifact_w
            if x1 < 0:
                artifact = artifact[-x1:, :]; x1 = 0
            if x2 > target_size[0]:
                artifact = artifact[:-(x2 - target_size[0]), :]; x2 = target_size[0]

            y1 = ys[i] - artifact_h // 2
            y2 = y1 + artifact_h
            if y1 < 0:
                artifact = artifact[:, -y1:]; y1 = 0
            if y2 > target_size[1]:
                artifact = artifact[:, :-(y2 - target_size[1])]; y2 = target_size[1]

            mask_final[x1:x2, y1:y2] = np.where(
                artifact > mask_final[x1:x2, y1:y2], artifact, mask_final[x1:x2, y1:y2]
            )
            i += 1
        except Exception:
            i += 1
            continue

    mask_final = np.invert(mask_final.astype(np.uint8))

    # Boost damage INTENSITY (how dark/light each already-damaged pixel is),
    # independent of size_variety (which only controls how BIG/how MANY
    # damage instances are). Many artifact types (scratches, spots, dirt)
    # have genuinely faint pixel values even at full size/count -- e.g.
    # spots average ~236/255, barely darker than clean at all. This pushes
    # already-damaged pixels further toward black; clean pixels (255)
    # are mathematically unaffected regardless of boost value, since
    # 255 - (255-255)*boost = 255 always.
    if intensity_boost != 1.0:
        mask_float = mask_final.astype(np.float32)
        mask_float = 255.0 - (255.0 - mask_float) * intensity_boost
        mask_final = np.clip(mask_float, 0, 255).astype(np.uint8)

    binarised = ((mask_final > 240) * 255).astype(np.uint8)
    return mask_final.astype(np.uint8), binarised


def add_procedural_scratches(mask, height, width, verbose=False):
    """Blend in fully procedural (Perlin-noise-based) scratch lines.
    These require NO source images at all -- real or synthetic -- so they
    are always 'safe' to include without pulling in any scan data."""
    num_extra_scratch = int(np.random.gamma(6, 2, 1)[0])
    for _ in range(num_extra_scratch):
        length = np.random.randint(10, high=max(height, width), dtype=int)
        try:
            scratch = line_scratch(np.array(length))
            sw, sh = scratch.shape[:2]
            if sw >= width or sh >= height:
                continue
            x1 = np.random.randint(0, width - sw)
            y1 = np.random.randint(0, height - sh)
            region = mask[x1:x1 + sw, y1:y1 + sh]
            mask[x1:x1 + sw, y1:y1 + sh] = np.minimum(region, np.invert(scratch.astype(np.uint8)))
        except Exception:
            continue
    if verbose:
        print(f"Added {num_extra_scratch} procedural scratch lines")
    return mask


if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description='Generate damage masks from ONLY classified synthetic patches (no scanned frames).'
    )
    parser.add_argument('--types', type=str, default='scratches,smut',
                         help='comma-separated subfolder names under /synthetic/, '
                              'e.g. scratches,smut,dirt,dots,hair,hair-short,lint,sprinkles,spots,stain')
    parser.add_argument('--height', type=int, default=1024)
    parser.add_argument('--width', type=int, default=1024)
    parser.add_argument('--min-count', type=int, default=3, help='min number of artifacts per type')
    parser.add_argument('--max-count', type=int, default=15, help='max number of artifacts per type')
    parser.add_argument('--size-variety-min', type=float, default=0.5,
                         help="minimum extra random size multiplier applied per artifact, independent of "
                              "the patch library's own area distribution -- lower values allow smaller "
                              "damage instances")
    parser.add_argument('--size-variety-max', type=float, default=1.8,
                         help="maximum extra random size multiplier applied per artifact -- higher values "
                              "allow larger damage instances")
    parser.add_argument('--intensity-boost', type=float, default=1.0,
                         help="darkens already-damaged pixels, independent of size/count -- some damage "
                              "types (spots, dirt, scratches) have genuinely faint pixel values even at "
                              "full size, e.g. spots average ~236/255. A value of 2.0-4.0 makes faint "
                              "damage clearly visible; 1.0 (default) leaves values unchanged. Clean "
                              "pixels are never affected by this, regardless of value.")
    parser.add_argument('--procedural-scratches', action='store_true',
                         help='also blend in fully procedural line scratches (no source image needed)')
    parser.add_argument('--n', type=int, default=1, help='how many masks to generate')
    parser.add_argument('--out-dir', type=str, default=None,
                         help='where to write masks. Defaults to <repo_root>/generated/. Set this explicitly '
                              'to generate straight into a per-type folder, e.g. --out-dir ../../data/masks/scratches '
                              'when running with --types scratches only, so different damage types land in '
                              'physically separate folders instead of one mixed pool.')
    parser.add_argument('--verbose', action='store_true')
    args = parser.parse_args()

    abs_path = os.path.abspath(os.path.dirname(__file__))
    synthetic_path = os.path.dirname(os.path.normpath(abs_path)) + '/synthetic/'
    out_dir = args.out_dir if args.out_dir else os.path.dirname(os.path.normpath(abs_path)) + '/generated/'
    if not out_dir.endswith('/'):
        out_dir += '/'
    os.makedirs(out_dir, exist_ok=True)

    types = [t.strip() for t in args.types.split(',') if t.strip()]

    per_type_dfs = {}
    for t in types:
        df = load_images(synthetic_path, t, verbose=args.verbose)
        df['Contour Area'] = df['Non-zero pixel area']
        per_type_dfs[t] = df
        print(f"Loaded {len(df)} '{t}' artifact patches from /synthetic/{t}/")

    per_type_counts = {t: (args.min_count, args.max_count) for t in types}

    for n in range(args.n):
        mask, binary_mask = build_mask(
            (args.height, args.width), per_type_dfs, per_type_counts, verbose=args.verbose,
            size_variety=(args.size_variety_min, args.size_variety_max),
            intensity_boost=args.intensity_boost,
        )

        if args.procedural_scratches:
            mask = add_procedural_scratches(mask, args.height, args.width, verbose=args.verbose)
            binary_mask = ((mask > 240) * 255).astype(np.uint8)

        uid = str(uuid.uuid4())[:8]
        tag = "_".join(types)
        cv.imwrite(out_dir + f'mask_{tag}_{uid}.png', mask)
        print(f"[{n+1}/{args.n}] Saved mask_{tag}_{uid}.png")

    print(f"Done. Masks written to {out_dir}")

Writing FilmDamageSimulator/damage_generator/generate_synthetic_only.py


In [4]:
TARGET_N_MASKS = 3000  # per type

TYPE_SETTINGS = {
    'scratches': {'min_count': 5, 'max_count': 35, 'size_min': 1.5, 'size_max': 1.5, 'boost': 2.5},
    'smut':      {'min_count': 5, 'max_count': 35, 'size_min': 1.5, 'size_max': 1.5, 'boost': 3.0},
    'spots':     {'min_count': 5, 'max_count': 35, 'size_min': 1.5, 'size_max': 1.5, 'boost': 8.0},
    'dirt':      {'min_count': 5, 'max_count': 35, 'size_min': 1.5, 'size_max': 1.5, 'boost': 3.0},
}

os.chdir('/kaggle/working/FilmDamageSimulator/damage_generator')

MASKS_DIRS = []
for damage_type, settings in TYPE_SETTINGS.items():
    type_dir = f'/kaggle/working/generated_masks/{damage_type}'
    os.makedirs(type_dir, exist_ok=True)
    existing = [f for f in os.listdir(type_dir) if f.startswith('mask_')]

    if len(existing) >= TARGET_N_MASKS:
        print(f'[{damage_type}] {len(existing)} masks already present, skipping.')
    else:
        needed = TARGET_N_MASKS - len(existing)
        print(f'[{damage_type}] generating {needed} masks (boost={settings["boost"]})...')
        abs_type_dir = os.path.abspath(type_dir)
        !python generate_synthetic_only.py --types {damage_type} \
            --height 128 --width 128 --min-count {settings['min_count']} --max-count {settings['max_count']} \
            --size-variety-min {settings['size_min']} --size-variety-max {settings['size_max']} \
            --intensity-boost {settings['boost']} \
            --n {needed} --out-dir {abs_type_dir} --verbose

    MASKS_DIRS.append(type_dir)

os.chdir('/kaggle/working')

for d in MASKS_DIRS:
    n = len([f for f in os.listdir(d) if f.startswith('mask_')]) if os.path.isdir(d) else 0
    print(f'{d}: {n} masks')

[scratches] generating 3000 masks (boost=2.5)...
Loading image overlay  scratch-0001.png
Loading image overlay  scratch-0002.png
Loading image overlay  scratch-0003.png
Loading image overlay  scratch-0004.png
Loading image overlay  scratch-0005.png
Loading image overlay  scratch-0006.png
Loading image overlay  scratch-0007.png
Loading image overlay  scratch-0008.png
Loading image overlay  scratch-0009.png
Loading image overlay  scratch-0010.png
Loading image overlay  scratch-0011.png
Loading image overlay  scratch-0012.png
Loading image overlay  scratch-0013.png
Loading image overlay  scratch-0014.png
Loading image overlay  scratch-0015.png
Loading image overlay  scratch-0016.png
Loading image overlay  scratch-0017.png
Loading image overlay  scratch-0018.png
Loading image overlay  scratch-0019.png
Loading image overlay  scratch-0020.png
Loading image overlay  scratch-0021.png
Loading image overlay  scratch-0022.png
Loading image overlay  scratch-0023.png
Loading image overlay  scratch-

In [5]:
# import matplotlib.pyplot as plt
# import cv2 as cv
# import random

# N_PREVIEW_SAMPLES = 4
# damage_types = list(TYPE_SETTINGS.keys())

# fig, axes = plt.subplots(len(damage_types), N_PREVIEW_SAMPLES, figsize=(3 * N_PREVIEW_SAMPLES, 3 * len(damage_types)))

# for row, (damage_type, mask_dir) in enumerate(zip(damage_types, MASKS_DIRS)):
#     mask_files = [f for f in os.listdir(mask_dir) if f.startswith('mask_')]
#     sample_masks = random.sample(mask_files, min(N_PREVIEW_SAMPLES, len(mask_files)))

#     for col, mf in enumerate(sample_masks):
#         mask_img = cv.imread(os.path.join(mask_dir, mf), cv.IMREAD_GRAYSCALE)
#         axes[row, col].imshow(mask_img, cmap='gray', vmin=0, vmax=255)
#         axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
#         if col == 0:
#             axes[row, col].set_ylabel(damage_type, fontsize=13)

# plt.tight_layout()
# plt.show()

In [6]:
# import torchvision.datasets as tvds

# VOC_ROOT = '/kaggle/working/voc_data'
# VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')

# if os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0:
#     print(f'VOC2012 already present at {VOC_JPEG_DIR}, skipping download.')
# else:
#     os.makedirs(VOC_ROOT, exist_ok=True)
#     _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)

# num_images = len([f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')])
# print(f'VOC2012 ready: {num_images} images at {VOC_JPEG_DIR}')


In [7]:
# %%writefile composite_damage.py
# """
# Composite a generated damage mask (from generate_synthetic_only.py or
# damage_generator.py) onto a clean target image, producing a damaged/clean
# training pair for restoration model training.

# The mask convention from this codebase: 255 = clean/undamaged, values toward
# 0 = damaged (dust, dirt, scratches etc).

# Two blend modes are supported:
#   - "screen" (default): LIGHTENS toward white at damaged pixels. This is the
#     physically realistic choice for most scratch/abrasion damage, where the
#     print's emulsion is scraped away and the lighter paper base shows
#     through -- old photo scratches are usually bright/white marks, not dark
#     ones.
#   - "multiply": DARKENS toward black at damaged pixels. More appropriate for
#     damage types that genuinely deposit dark material (soot/smut, heavy
#     dirt, mold staining) rather than abrading the surface.

# Since a single generated mask can currently mix multiple damage types
# (e.g. scratches + smut) without tracking which pixel came from which type,
# this is a per-composite choice rather than automatic per-pixel selection.
# If your mask pool separates damage types into different files (e.g. by
# generating scratches and smut as separate mask batches), you can composite
# each with the blend mode that suits it and merge afterward, rather than
# using one blend mode for a mixed mask.

# Usage:
#     python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png
#     python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend multiply
# """

# import argparse
# import cv2 as cv
# import numpy as np


# def composite(clean_img, mask_img, blend="screen"):
#     if clean_img.shape[:2] != mask_img.shape[:2]:
#         mask_img = cv.resize(mask_img, (clean_img.shape[1], clean_img.shape[0]), interpolation=cv.INTER_LINEAR)

#     mask_norm = mask_img.astype(np.float32) / 255.0
#     if clean_img.ndim == 3 and mask_norm.ndim == 2:
#         mask_norm = mask_norm[:, :, None]

#     clean_f = clean_img.astype(np.float32)

#     if blend == "screen":
#         # Lightens toward white at damaged (low-mask) pixels.
#         damaged = 255.0 - (255.0 - clean_f) * mask_norm
#     elif blend == "multiply":
#         # Darkens toward black at damaged (low-mask) pixels.
#         damaged = clean_f * mask_norm
#     else:
#         raise ValueError(f"Unknown blend mode '{blend}', expected 'screen' or 'multiply'")

#     return np.clip(damaged, 0, 255).astype(np.uint8)


# if __name__ == '__main__':
#     parser = argparse.ArgumentParser(description='Composite a damage mask onto a clean image.')
#     parser.add_argument('--clean', required=True, help='path to the clean input image')
#     parser.add_argument('--mask', required=True, help='path to the generated grayscale damage mask')
#     parser.add_argument('--out', required=True, help='path to write the damaged output image')
#     parser.add_argument('--blend', choices=['screen', 'multiply'], default='screen',
#                          help="'screen' (default) produces light/white damage marks; "
#                               "'multiply' produces dark damage marks")
#     args = parser.parse_args()

#     clean_img = cv.imread(args.clean, cv.IMREAD_UNCHANGED)
#     mask_img = cv.imread(args.mask, cv.IMREAD_GRAYSCALE)

#     damaged = composite(clean_img, mask_img, blend=args.blend)
#     cv.imwrite(args.out, damaged)
#     print(f"Wrote damaged image to {args.out} (blend={args.blend})")


In [8]:
 # import matplotlib.pyplot as plt
 # import cv2 as cv
 # import random
 # from composite_damage import composite

 # clean_files = [f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')]
 # sample_clean_files = random.sample(clean_files, 4)

 # mask_files = []
 # for d in MASKS_DIRS:
 #     mask_files += [os.path.join(d, f) for f in os.listdir(d) if f.startswith('mask_')]
 # sample_mask_files = random.sample(mask_files, 4)

 # fig, axes = plt.subplots(4, 2, figsize=(6, 12))
 # for i, (clean_fname, mask_path) in enumerate(zip(sample_clean_files, sample_mask_files)):
 #     clean_img = cv.imread(os.path.join(VOC_JPEG_DIR, clean_fname))
 #     mask_img = cv.imread(mask_path, cv.IMREAD_GRAYSCALE)
 #     damaged_img = composite(clean_img, mask_img)
 #     axes[i, 0].imshow(cv.cvtColor(clean_img, cv.COLOR_BGR2RGB)); axes[i,0].set_title('Clean'); axes[i,0].axis('off')
 #     axes[i, 1].imshow(cv.cvtColor(damaged_img, cv.COLOR_BGR2RGB)); axes[i,1].set_title('Damaged'); axes[i,1].axis('off')
 # plt.tight_layout()
 # plt.show()

In [9]:
import shutil, stat, os

def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

shutil.rmtree("/kaggle/working/FilmDamageSimulator", onerror=remove_readonly)

In [10]:
import shutil,os

shutil.make_archive("/kaggle/working/generated_masks", "zip", "/kaggle/working/generated_masks")
print('Zipped to /kaggle/working/generated_masks.zip')

assert os.path.exists("/kaggle/working/generated_masks.zip"), 'Zip was not created -- aborting, NOT deleting source folder'
zip_size_mb = os.path.getsize("/kaggle/working/generated_masks.zip") / (1024 * 1024)
assert zip_size_mb > 0.01, f'Zip file suspiciously small ({zip_size_mb:.3f} MB) -- aborting, NOT deleting source folder'

print(f'Verified: {"/kaggle/working/generated_masks.zip"} exists and is {zip_size_mb:.1f} MB')
shutil.rmtree("/kaggle/working/generated_masks")
print(f'Deleted {"/kaggle/working/generated_masks"}')

Zipped to /kaggle/working/generated_masks.zip
Verified: /kaggle/working/generated_masks.zip exists and is 41.1 MB
Deleted /kaggle/working/generated_masks


In [11]:
# import shutil,os

# shutil.make_archive("/kaggle/working/generated_masks", "zip", "/kaggle/working/generated_masks")
# print('Zipped to /kaggle/working/generated_masks.zip')

# assert os.path.exists("/kaggle/working/generated_masks.zip"), 'Zip was not created -- aborting, NOT deleting source folder'
# zip_size_mb = os.path.getsize("/kaggle/working/generated_masks.zip") / (1024 * 1024)
# assert zip_size_mb > 0.01, f'Zip file suspiciously small ({zip_size_mb:.3f} MB) -- aborting, NOT deleting source folder'